# Quickstart: RT-J on RelBench

Evaluate the released **RT-J** checkpoint on one RelBench task, `rel-f1/driver-dnf`
(will an F1 driver fail to finish the race?), with no training. The model predicts
every test row from a context sampled around it, and we score the predictions with AUROC.

*Runtime -> Change runtime type -> **GPU**.* A few minutes end to end.

## 1. Install

In [ ]:
!pip install -q uv && uv pip install -q --system relational-transformer
print("setup done")

## 2. Data and model

RelBench, already preprocessed into RT's tensor format, is on the Hub. One database is enough.

In [ ]:
import torch
from huggingface_hub import snapshot_download

from rt import RelationalTransformer

pre_dir = snapshot_download(
    "stanford-star/relbench-preprocessed", repo_type="dataset", allow_patterns="rel-f1/*"
)
model = RelationalTransformer.from_pretrained("stanford-star/rt-j", device="cuda").to(torch.bfloat16)
cfg = model.config
print("embedder:", cfg["embedder"], "| d_text:", cfg["d_text"])

## 3. Predict

The evaluator samples each test row's context and runs the model over it. The context
knobs below are RT-J's defaults: 8192 cells in total, 256 around the seed row, 32 nodes per BFS level.

In [ ]:
from rt.data import get_tasks
from rt.eval import build_evaluator

tasks = get_tasks(pre_dir, [("rel-f1", "driver-dnf")], ("test",))
ev = build_evaluator(
    tasks, pre_dir,
    embedder=cfg["embedder"], d_text=cfg["d_text"], device="cuda",
    ctx_size_list=[8192], local_ctx_size=256, bfs_width=32, prefer_latest=True,
    num_walks=10_000, walk_length=20, tokens_per_gpu=2**18,
    items_per_task=10_000_000, num_workers=2, prefetch_factor=2,
    shuffle_seed=0, context_seed=0, mmap_populate=True, vector_db_path=None,
    db_cutoff=None,
)
_task, _ctx, labels, out, _n = next(iter(ev.evaluate_raw([(model, "")], [8192])))
probs = torch.sigmoid(torch.tensor(out[""], dtype=torch.float32)).numpy()
print(f"predicted {len(labels)} test rows")

## 4. Score

In [ ]:
from sklearn.metrics import roc_auc_score

print(f"rel-f1/driver-dnf  AUROC = {roc_auc_score(labels > 0, probs):.4f}")

## Next

- Swap the `(db, task)` pair for any other RelBench task, or download more of `relbench-preprocessed`.
- Several `ctx_size_list` sizes score for the price of one; `local_ctx_size` and `bfs_width` shape the context ([examples/eval](https://github.com/stanford-star/relational-transformer/blob/main/examples/eval/README.md)).
- Your own database: [notebooks/byod.ipynb](https://colab.research.google.com/github/stanford-star/relational-transformer/blob/main/notebooks/byod.ipynb).